In [0]:
tables = [row.name for row in spark.catalog.listTables("samples.tpch")]

for table in tables:
    src = f"samples.tpch.{table}"
    tgt = f"cmoore_user.tpch_semantic_views.{table}"
    spark.sql(f"CREATE OR REPLACE TABLE {tgt} AS SELECT * FROM {src}")

In [0]:
import yaml

# Helper: get columns and types for each table
def get_table_schema(table):
    desc = spark.sql(f"DESCRIBE cmoore_user.tpch_semantic_views.{table}").toPandas()
    return [
        {"name": row["col_name"], "type": row["data_type"]}
        for _, row in desc.iterrows()
        if row["col_name"] not in ("#", "") and not row["col_name"].startswith("#")
    ]

# Helper: suggest dimensions and measures
def suggest_semantics(table, columns):
    dims, meas = [], []
    for col in columns:
        name, typ = col["name"], col["type"]
        if typ in ("int", "bigint", "long", "decimal", "double", "float"):
            if name.startswith("key") or name.endswith("key") or name.startswith("id"):
                dims.append(name)
            else:
                meas.append({"name": f"sum_{name}", "expression": f"SUM({name})"})
                meas.append({"name": f"avg_{name}", "expression": f"AVG({name})"})
        elif typ in ("string", "date", "timestamp"):
            dims.append(name)
    return dims, meas

# AI-generated table descriptions (example, replace with actual AI if available)
ai_table_descriptions = {
    "customer": "Represents customers who place orders. Contains demographic and account information.",
    "lineitem": "Details of individual items within an order, including quantity, price, and shipping info.",
    "nation": "List of nations for customer and supplier locations.",
    "orders": "Records of customer orders, including order date and status.",
    "part": "Catalog of products available for sale.",
    "partsupp": "Associates parts with suppliers and their supply costs.",
    "region": "Geographical regions grouping nations.",
    "supplier": "Information about suppliers providing parts."
}

# Primary and foreign key definitions (example, adjust as needed)
table_keys = {
    "customer": {
        "primary_key": ["c_custkey"],
        "foreign_keys": [
            {"column": "c_nationkey", "ref_table": "nation", "ref_column": "n_nationkey"}
        ]
    },
    "lineitem": {
        "primary_key": ["l_orderkey", "l_partkey", "l_suppkey"],
        "foreign_keys": [
            {"column": "l_orderkey", "ref_table": "orders", "ref_column": "o_orderkey"},
            {"column": "l_partkey", "ref_table": "part", "ref_column": "p_partkey"},
            {"column": "l_suppkey", "ref_table": "supplier", "ref_column": "s_suppkey"}
        ]
    },
    "nation": {
        "primary_key": ["n_nationkey"],
        "foreign_keys": [
            {"column": "n_regionkey", "ref_table": "region", "ref_column": "r_regionkey"}
        ]
    },
    "orders": {
        "primary_key": ["o_orderkey"],
        "foreign_keys": [
            {"column": "o_custkey", "ref_table": "customer", "ref_column": "c_custkey"}
        ]
    },
    "part": {
        "primary_key": ["p_partkey"],
        "foreign_keys": []
    },
    "partsupp": {
        "primary_key": ["ps_partkey", "ps_suppkey"],
        "foreign_keys": [
            {"column": "ps_partkey", "ref_table": "part", "ref_column": "p_partkey"},
            {"column": "ps_suppkey", "ref_table": "supplier", "ref_column": "s_suppkey"}
        ]
    },
    "region": {
        "primary_key": ["r_regionkey"],
        "foreign_keys": []
    },
    "supplier": {
        "primary_key": ["s_suppkey"],
        "foreign_keys": [
            {"column": "s_nationkey", "ref_table": "nation", "ref_column": "n_nationkey"}
        ]
    }
}

# Final script to implement changes on tables
for table in tables:
    description = ai_table_descriptions.get(table, f"AI-generated description for {table}.")
    spark.sql(f"COMMENT ON TABLE cmoore_user.tpch_semantic_views.{table} IS '{description}'")
    keys = table_keys.get(table, {"primary_key": [], "foreign_keys": []})

    # Set all PK and FK columns to NOT NULL
    not_null_cols = set(keys["primary_key"])
    not_null_cols.update(fk["column"] for fk in keys["foreign_keys"])
    for col in not_null_cols:
        spark.sql(f"ALTER TABLE cmoore_user.tpch_semantic_views.{table} ALTER COLUMN {col} SET NOT NULL")

    # Drop existing PK constraints if they exist
    spark.sql(f"ALTER TABLE cmoore_user.tpch_semantic_views.{table} DROP CONSTRAINT IF EXISTS pk_{table}")

    # Add PK constraint (compound if needed)
    if keys["primary_key"]:
        pk_cols = ", ".join(keys["primary_key"])
        spark.sql(
            f"ALTER TABLE cmoore_user.tpch_semantic_views.{table} "
            f"ADD CONSTRAINT pk_{table} PRIMARY KEY ({pk_cols})"
        )

# Add FK constraints after all PKs are set
for table in tables:
    keys = table_keys.get(table, {"primary_key": [], "foreign_keys": []})
    for fk in keys["foreign_keys"]:
        # Drop existing FK constraint if it exists
        spark.sql(
            f"ALTER TABLE cmoore_user.tpch_semantic_views.{table} "
            f"DROP CONSTRAINT IF EXISTS fk_{table}_{fk['column']}_to_{fk['ref_table']}"
        )
        # Add FK constraint
        spark.sql(
            f"ALTER TABLE cmoore_user.tpch_semantic_views.{table} "
            f"ADD CONSTRAINT fk_{table}_{fk['column']}_to_{fk['ref_table']} "
            f"FOREIGN KEY ({fk['column']}) REFERENCES cmoore_user.tpch_semantic_views.{fk['ref_table']}({fk['ref_column']})"
        )

In [0]:
import yaml

metric_view_yaml = {
    "version": "1.1",
    "name": "orders_metric_view",
    "source": "cmoore_user.tpch_semantic_views.orders",
    "joins": [
        {
            "name": "customer",
            "source": "cmoore_user.tpch_semantic_views.customer",
            "on": "source.o_custkey = customer.c_custkey",
            "joins": [
                {
                    "name": "nation",
                    "source": "cmoore_user.tpch_semantic_views.nation",
                    "on": "customer.c_nationkey = nation.n_nationkey",
                    "joins": [
                        {
                            "name": "region",
                            "source": "cmoore_user.tpch_semantic_views.region",
                            "on": "nation.n_regionkey = region.r_regionkey"
                        }
                    ]
                }
            ]
        },
        {
            "name": "lineitem",
            "source": "cmoore_user.tpch_semantic_views.lineitem",
            "on": "source.o_orderkey = lineitem.l_orderkey",
            "joins": [
                {
                    "name": "part",
                    "source": "cmoore_user.tpch_semantic_views.part",
                    "on": "lineitem.l_partkey = part.p_partkey"
                },
                {
                    "name": "supplier",
                    "source": "cmoore_user.tpch_semantic_views.supplier",
                    "on": "lineitem.l_suppkey = supplier.s_suppkey",
                    "joins": [
                        {
                            "name": "nation",
                            "source": "cmoore_user.tpch_semantic_views.nation",
                            "on": "supplier.s_nationkey = nation.n_nationkey",
                            "joins": [
                                {
                                    "name": "region",
                                    "source": "cmoore_user.tpch_semantic_views.region",
                                    "on": "nation.n_regionkey = region.r_regionkey"
                                }
                            ]
                        }
                    ]
                }
            ]
        }
    ],
    "dimensions": [
        {"name": "order_clerk", "expr": "o_clerk"},
        {"name": "order_status", "expr": "o_orderstatus"},
        {"name": "order_date", "expr": "o_orderdate"},
        {"name": "customer_name", "expr": "customer.c_name"},
        {"name": "customer_nation", "expr": "customer.nation.n_name"},
        {"name": "customer_region", "expr": "customer.nation.region.r_name"},
        {"name": "part_name", "expr": "lineitem.part.p_name"},
        {"name": "part_type", "expr": "lineitem.part.p_type"},
        {"name": "supplier_name", "expr": "lineitem.supplier.s_name"},
        {"name": "supplier_nation", "expr": "lineitem.supplier.nation.n_name"},
        {"name": "supplier_region", "expr": "lineitem.supplier.nation.region.r_name"}
    ],
    "measures": [
        {"name": "order_count", "expression": "COUNT(*)"},
        {"name": "total_price", "expression": "SUM(o_totalprice)"},
        {"name": "total_quantity", "expression": "SUM(lineitem.l_quantity)"},
        {"name": "total_extended_price", "expression": "SUM(lineitem.l_extendedprice)"}
    ]
}

yaml_str = yaml.dump(metric_view_yaml, sort_keys=False)
print(yaml_str)

In [0]:
for table in tables:
    spark.sql(f"ALTER TABLE cmoore_user.tpch_semantic_views.{table} CLUSTER BY AUTO")
    spark.sql(f"OPTIMIZE cmoore_user.tpch_semantic_views.{table}")
    spark.sql(f"ANALYZE TABLE cmoore_user.tpch_semantic_views.{table} COMPUTE STATISTICS FOR ALL COLUMNS")